In [ ]:
import subprocess
import uuid
import time
import os

def download_video(url, title, retries=3):
    outname = f"video_{uuid.uuid4()}.mp4"
    cookie_file = os.getenv("YT_COOKIE_FILE", "cookies.txt")

    # Always use cookies.txt — browser DB cannot be accessed on Windows
    if not os.path.isfile(cookie_file):
        print(f"❌ Missing cookie file: {cookie_file}")
        print("    You must export cookies using the browser extension:")
        print("    1. Install 'Get cookies.txt' extension")
        print("    2. Open youtube.com")
        print("    3. Export cookies to cookies.txt")
        return None

    cmd = [
        "yt-dlp",
        "--cookies", cookie_file,      # Always use exported cookies
        "-f", "b[ext=mp4]",            # Safe MP4 selector
        "-o", outname,
        url
    ]

    for attempt in range(1, retries + 1):
        print(f"⬇️  Downloading: {title}  (Attempt {attempt}/{retries})")

        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"✅ Downloaded: {title}")
            return outname

        except subprocess.CalledProcessError as e:
            stderr = e.stderr.strip()
            print(f"❌ Download failed for {title}")
            print("Reason:", stderr)

            if attempt < retries:
                print("🔁 Retrying...")
                time.sleep(2)
            else:
                print(f"⛔ Skipping {title} after {retries} failed attempts.")
                return None


In [ ]:
import pytesseract
from moviepy.editor import VideoFileClip
from PIL import Image
import os
import re

FAILED_OCR = []


def extract_summary_from_video(url, title):
    filepath = download_video(url, title)
    if filepath is None:
        print(f"⛔ Skipping OCR for {title} (download failed)")
        return {}

    try:
        clip = VideoFileClip(filepath)
        end_time = clip.duration
        start_time = max(0, end_time - 10)

        frame = clip.get_frame(start_time + 5)
        img = Image.fromarray(frame)

        raw_text = pytesseract.image_to_string(img)

        pattern = r"(.+?):\s*(\d+)\s*goal[s]?,\s*(\d+)\s*assist[s]?"
        stats = {}

        for line in raw_text.split("\n"):
            match = re.search(pattern, line.strip())
            if match:
                player = match.group(1).strip()
                goals = int(match.group(2))
                assists = int(match.group(3))
                stats[player] = {"goals": goals, "assists": assists}

        clip.close()
        os.remove(filepath)

        print(f"📊 OCR Success: {title}")
        return stats

    except Exception as e:
        print(f"❌ OCR failed for {title}")        
        print("Reason:", str(e))
        FAILED_OCR.append({"title": title, "url": url, "reason": str(e)})
        return {}


In [ ]:
from collections import defaultdict

def aggregate_player_stats(video_items):
    """
    video_items = list of dicts:
    [
        {"url": "...", "title": "..."},
        ...
    ]
    """
    totals = defaultdict(lambda: {"goals": 0, "assists": 0, "games": []})

    for item in video_items:
        url = item["url"]
        title = item["title"]

        print(f"\n🔍 Processing video: {title}")

        summary = extract_summary_from_video(url, title)

        if not summary:
            print(f"⚠️ No stats extracted for {title}")
            FAILED_OCR.append({"title": title, "url": url, "reason": "No stats extracted"})
            continue

        for player, s in summary.items():
            totals[player]["goals"] += s["goals"]
            totals[player]["assists"] += s["assists"]
            totals[player]["games"] += [title]

        print(f"🏁 Completed: {title}")

    return dict(totals)


In [ ]:
import requests
from datetime import datetime

def get_uploaded_video_urls_filtered(api_key, channel_id, 
                                     start_date=None, end_date=None, 
                                     keywords=None):
    """
    Fetches uploaded videos and filters by date range + title keywords.
    
    Returns list of dicts:
    [
        {"url": "...", "title": "...", "published": datetime_obj},
        ...
    ]
    """

    # Step 1: Get uploads playlist ID
    url = "https://www.googleapis.com/youtube/v3/channels"
    params = {
        "part": "contentDetails",
        "id": channel_id,
        "key": api_key
    }
    r = requests.get(url, params=params).json()
    uploads_playlist = r["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

    # Convert dates
    if start_date:
        start_date = datetime.fromisoformat(start_date)
    if end_date:
        end_date = datetime.fromisoformat(end_date)

    # Step 2: Fetch videos
    video_items = []
    next_page = None

    while True:
        playlist_url = "https://www.googleapis.com/youtube/v3/playlistItems"
        playlist_params = {
            "part": "contentDetails,snippet",
            "playlistId": uploads_playlist,
            "maxResults": 50,
            "key": api_key
        }
        if next_page:
            playlist_params["pageToken"] = next_page

        playlist_response = requests.get(playlist_url, params=playlist_params).json()

        for item in playlist_response["items"]:
            video_id = item["contentDetails"]["videoId"]
            title = item["snippet"]["title"]
            published = datetime.fromisoformat(
                item["contentDetails"]["videoPublishedAt"].replace("Z", "")
            )

            # === Apply filters ===

            # Date range filter
            if start_date and published < start_date:
                continue
            if end_date and published > end_date:
                continue

            # Keyword filter
            if keywords:
                title_lower = title.lower()
                if not any(k.lower() in title_lower for k in keywords):
                    continue

            # Passed all filters → add full item
            video_items.append({
                "url": f"https://www.youtube.com/watch?v={video_id}",
                "title": title,
                "published": published
            })

        next_page = playlist_response.get("nextPageToken")
        if not next_page:
            break

    return video_items


In [ ]:
FIX_KEYWORDS = ["fixed", "corrected", "updated", "v2", "version 2", "reupload", "final"]

def is_fixed_version(title):
    t = title.lower()
    return any(k in t for k in FIX_KEYWORDS)

def dedupe_fixed_videos(video_items):
    grouped = {}

    for item in video_items:
        title = item["title"].lower()

        # Use the part before "highlights" as the match identifier
        if "highlights" in title:
            match_key = title.split("highlights")[0].strip()
        else:
            match_key = title

        if match_key not in grouped:
            grouped[match_key] = item
        else:
            # Replace original with fixed version
            if is_fixed_version(title):
                grouped[match_key] = item

    return list(grouped.values())



In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file

API_KEY = os.getenv("YT_API_KEY")
CHANNEL_ID = os.getenv("YT_CHANNEL_ID")

urls = get_uploaded_video_urls_filtered(
    api_key=API_KEY,
    channel_id=CHANNEL_ID,
    start_date="2025-09-01",
    end_date="2026-08-31",
    keywords=["highlights"]
)

urls = dedupe_fixed_videos(urls)

print("the number of filtered video URLs is:", len(urls))



In [ ]:
totals = aggregate_player_stats(urls)


In [ ]:
def print_leaderboard(totals):
    print("\n=== Player Leaderboard ===")
    sorted_players = sorted(
        totals.items(),
        key=lambda x: (x[1]["goals"], x[1]["assists"]),
        reverse=True
    )

    for player, stats in sorted_players:
        print(f"{player:10}  Goals: {stats['goals']:2}  Assists: {stats['assists']:2}  Games Detected: {len(stats['games'])}")


In [ ]:
print_leaderboard(totals)

In [ ]:
print("\n=== Videos with failed OCR ===")
for item in FAILED_OCR:
    print(f"- {item['title']} ({item['url']})")


In [ ]:
from rapidfuzz import process

KNOWN_PLAYERS = os.getenv("KNOWN_PLAYERS", [])

def clean_totals_with_fuzzy_matching(totals, known_players=KNOWN_PLAYERS, threshold=85):
    """
    Cleans OCR totals using RapidFuzz fuzzy matching.
    Merges mistyped names into canonical names.
    Returns cleaned totals + a log of corrections.
    """

    cleaned = {player: {"goals": 0, "assists": 0, "games": []} for player in known_players}
    corrections = []

    for name, stats in totals.items():
        # Fuzzy match name to known players

        best_match, score, _ = process.extractOne(name, known_players)

        if score >= threshold:
            # Merge stats into canonical name
            cleaned[best_match]["goals"] += stats["goals"]
            cleaned[best_match]["assists"] += stats["assists"]
            cleaned[best_match]["games"] += stats["games"]

            if best_match.lower() != name.lower():
                corrections.append((name, best_match, stats["games"], score))
        else:
            # If no good match, keep the name as-is
            cleaned[name] = stats
            corrections.append((name, name, stats["games"], score))

    return cleaned, corrections



In [ ]:
cleaned_totals, corrections = clean_totals_with_fuzzy_matching(totals, threshold=61)

print("=== Cleaned Totals ===")
print(cleaned_totals)

print("\n=== Name Corrections Applied ===")
for old, new, games, score in corrections:
    print(f"{old} → {new}  -> {games}(score: {score})")


In [ ]:
print_leaderboard(cleaned_totals)